# CSCI 4930: Group Project
Mark Evers  
Marlon Williams  
John Sepulvuda  
Kyle

### Install

In [ ]:
# %pip install -r requirements.txt

### Imports

In [ ]:
import gdown
import numpy as np
import os
import pandas as pd

### Download dataset into workspace

In [ ]:
# check if the file exists
if not os.path.exists("data/ODC_CRIME_TRAFFICACCIDENTS5YR_P_7255667086186507966.csv"):
    # create the data directory if it doesn't exist
    os.makedirs("data", exist_ok=True)
    # download the file from Google Drive
    gdown.download("https://drive.google.com/uc?id=1wcvSB8gTEAzbq2ARRhjGJGyoMlP-Rby3", "data/ODC_CRIME_TRAFFICACCIDENTS5YR_P_7255667086186507966.csv", quiet=False)

### Load the data

In [ ]:
keep_cols = [
    "offense_code",
    "offense_code_extension",
    "top_traffic_accident_offense",
    "first_occurrence_date",
    "district_id",
    "precinct_id",
    "neighborhood_id",
    "bicycle_ind",
    "pedestrian_ind",
    "HARMFUL_EVENT_SEQ_1",
    "HARMFUL_EVENT_SEQ_2",
    "HARMFUL_EVENT_SEQ_3",
    "road_location",
    "ROAD_DESCRIPTION",
    "ROAD_CONTOUR",
    "ROAD_CONDITION",
    "LIGHT_CONDITION",
    "TU1_VEHICLE_TYPE",
    "TU1_TRAVEL_DIRECTION",
    "TU1_VEHICLE_MOVEMENT",
    "TU1_DRIVER_ACTION",
    "TU1_DRIVER_HUMANCONTRIBFACTOR",
    "TU1_PEDESTRIAN_ACTION",
    "TU2_VEHICLE_TYPE",
    "TU2_TRAVEL_DIRECTION",
    "TU2_VEHICLE_MOVEMENT",
    "TU2_DRIVER_ACTION",
    "TU2_DRIVER_HUMANCONTRIBFACTOR",
    "TU2_PEDESTRIAN_ACTION",
    "geo_lon",
    "geo_lat",
    # these are the target variables
    "SERIOUSLY_INJURED",
    "FATALITIES"
]
df = pd.read_csv("data/ODC_CRIME_TRAFFICACCIDENTS5YR_P_7255667086186507966.csv", usecols=keep_cols)
print(f"Total rows: {len(df)}")

Total rows: 282244


/tmp/ipykernel_40124/3599790975.py:37: DtypeWarning: Columns (0: road_location) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/ODC_CRIME_TRAFFICACCIDENTS5YR_P_7255667086186507966.csv", usecols=keep_cols)


### Drop rows with missing values

In [ ]:
total_before = len(df)
print(f"Total rows before dropping null values: {total_before}")

n_before = len(df)
df = df[(~df.SERIOUSLY_INJURED.isnull()) & (~df.FATALITIES.isnull())]
n_after = len(df)
print(f"Dropped {n_before - n_after} rows with null values in column SERIOUSLY_INJURED and FATALITIES.")

n_before = len(df)
df = df[(~df.ROAD_DESCRIPTION.isnull()) & (~df.ROAD_CONTOUR.isnull()) & (~df.ROAD_CONDITION.isnull())]
n_after = len(df)
print(f"Dropped {n_before - n_after} rows with null values in columns ROAD_DESCRIPTION, ROAD_CONTOUR, and ROAD_CONDITION.")

n_before = n_after
df = df[(~df.HARMFUL_EVENT_SEQ_1.isnull())]
n_after = len(df)
print(f"Dropped {n_before - n_after} rows with null values in column HARMFUL_EVENT_SEQ_1.")

n_before = n_after
df = df[(~df.TU1_TRAVEL_DIRECTION.isnull())]
n_after = len(df)
print(f"Dropped {n_before - n_after} rows with null values in column TU1_TRAVEL_DIRECTION.")

n_before = n_after
df = df[(~df.geo_lon.isnull())]
n_after = len(df)
print(f"Dropped {n_before - n_after} rows with null values in column geo_lon and geo_lat.")

n_before = n_after
df = df[(~df.TU1_DRIVER_ACTION.isnull())]
n_after = len(df)
print(f"Dropped {n_before - n_after} rows with null values in column TU1_DRIVER_ACTION.")

n_before = n_after
df = df[(~df.TU1_DRIVER_HUMANCONTRIBFACTOR.isnull())]
n_after = len(df)
print(f"Dropped {n_before - n_after} rows with null values in column TU1_DRIVER_HUMANCONTRIBFACTOR.")

total_after = len(df)
print(f"Total rows after dropping null values: {total_after} ({total_before - total_after} rows dropped in total).")


Total rows before dropping null values: 282244
Dropped 1540 rows with null values in column SERIOUSLY_INJURED and FATALITIES.
Dropped 3066 rows with null values in columns ROAD_DESCRIPTION, ROAD_CONTOUR, and ROAD_CONDITION.
Dropped 220 rows with null values in column HARMFUL_EVENT_SEQ_1.
Dropped 10029 rows with null values in column TU1_TRAVEL_DIRECTION.
Dropped 10499 rows with null values in column geo_lon and geo_lat.
Dropped 2717 rows with null values in column TU1_DRIVER_ACTION.
Total rows after dropping null values: 254173 (28071 rows dropped in total).


In [ ]:
print("Trimming whitespace...")
df["top_traffic_accident_offense"] = df["top_traffic_accident_offense"].str.strip()
df["district_id"] = df["district_id"].str.strip()

Trimming whitespace...


In [ ]:
len(df[df["TU1_DRIVER_HUMANCONTRIBFACTOR"].isnull()])
# df["neighborhood_id"].unique()

2692

In [ ]:
print("Filling in missing values...")
df["geo_lon"] = df["geo_lon"].fillna(0)
df["geo_lat"] = df["geo_lat"].fillna(0)
df["district_id"] = df["district_id"].fillna(0)
df["precinct_id"] = df["precinct_id"].fillna(0)
df["neighborhood_id"] = df["neighborhood_id"].fillna("Unknown")
df["bicycle_ind"] = df["bicycle_ind"].fillna(0)
df["pedestrian_ind"] = df["pedestrian_ind"].fillna(0)
df["HARMFUL_EVENT_SEQ_1"] = df["HARMFUL_EVENT_SEQ_1"].fillna("None")
df["HARMFUL_EVENT_SEQ_2"] = df["HARMFUL_EVENT_SEQ_2"].fillna("None")
df["HARMFUL_EVENT_SEQ_3"] = df["HARMFUL_EVENT_SEQ_3"].fillna("None")
df["road_location"] = df["road_location"].fillna("On Roadway")
df["ROAD_DESCRIPTION"] = df["ROAD_DESCRIPTION"].replace("  ", "UNDER INVESTIGATION")
df["ROAD_CONTOUR"] = df["ROAD_CONTOUR"].replace("  ", "Unknown")
df["ROAD_CONDITION"] = df["ROAD_CONDITION"].replace("  ", "UNDER INVESTIGATION")
df["LIGHT_CONDITION"] = df["LIGHT_CONDITION"].replace("  ", "UNDER INVESTIGATION")
df["LIGHT_CONDITION"] = df["LIGHT_CONDITION"].fillna("UNDER INVESTIGATION")
df["TU1_VEHICLE_TYPE"] = df["TU1_VEHICLE_TYPE"].fillna("UNDER INVESTIGATION")
df["TU1_VEHICLE_TYPE"] = df["TU1_VEHICLE_TYPE"].replace("  ", "UNDER INVESTIGATION")
df = df[df["TU1_VEHICLE_TYPE"] != "0"]
df = df[df["TU1_VEHICLE_TYPE"] != "UNK"]
df["TU1_PEDESTRIAN_ACTION"] = df["TU1_PEDESTRIAN_ACTION"].fillna("None")
df["TU2_VEHICLE_TYPE"] = df["TU2_VEHICLE_TYPE"].fillna("None")
df["TU2_TRAVEL_DIRECTION"] = df["TU2_TRAVEL_DIRECTION"].fillna("None")
df["TU2_VEHICLE_MOVEMENT"] = df["TU2_VEHICLE_MOVEMENT"].fillna("None")
df["TU2_DRIVER_ACTION"] = df["TU2_DRIVER_ACTION"].fillna("No Contributing Action")
df["TU2_DRIVER_HUMANCONTRIBFACTOR"] = df["TU2_DRIVER_HUMANCONTRIBFACTOR"].fillna("No Apparent Contributing Factor")
df["TU2_PEDESTRIAN_ACTION"] = df["TU2_PEDESTRIAN_ACTION"].fillna("None")

Filling in missing values...


In [ ]:
df.to_csv("data/processed_crime_traffic_accidents.csv", index=False)